In [ ]:
import os

root = "/kaggle/input"

for current, dirs, files in os.walk(root):
    depth = current[len(root):].count(os.sep)

    print(f"\n目录：{current}")
    print("子目录：", dirs[:10])
    print("文件示例：", files[:5])

    # 只检查目录结构，不遍历每张图片
    if depth >= 4:
        dirs[:] = []

In [ ]:
from pathlib import Path


DATA_ROOT = Path(
    "/kaggle/input/datasets/dididoudou/glaucoma-data"
)


for folder in [
    "train_1",
    "train_2",
    "train_3",
    "train_4",
    "train_5",
    "valid",
    "test"
]:

    path = DATA_ROOT / folder

    print("\n======", folder, "======")

    if path.exists():

        print("第一层:")
        print(list(path.iterdir())[:3])

        jpgs = list(path.rglob("*.jpg")) + list(path.rglob("*.JPG"))

        print("图片数量:", len(jpgs))

    else:

        print("不存在")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    print(files[:5])

In [ ]:
import os

for x in os.listdir("/kaggle/input/datasets/dididoudou"):
    print(x)

In [ ]:
import os

ROOT="/kaggle/input/datasets/dididoudou/glaucoma-first-data"

for root, dirs, files in os.walk(ROOT):

    print("\n目录:",root)

    if files:
        print("文件数量:",len(files))
        print("示例:",files[:5])

In [ ]:
print("Kaggle运行正常！")

In [ ]:
import os

base = "/kaggle/input/datasets/dididoudou"

print("===== 已挂载的数据集 =====")

if os.path.exists(base):
    print(os.listdir(base))
else:
    print("没有找到该目录，请检查 /kaggle/input 下的路径")

print("\n===== 各数据集中的文件夹和 CSV =====")

for name in [
    "glaucoma",
    "glaucoma-data",
    "glaucoma-first",
    "glaucoma-first-data",
]:
    path = os.path.join(base, name)

    print(f"\n--- {name} ---")

    if not os.path.exists(path):
        print("当前路径不存在")
        continue

    print("子目录：", os.listdir(path)[:15])

    for f in os.listdir(path):
        full_path = os.path.join(path, f)

        if f.lower().endswith(".csv"):
            print("CSV：", f)

        elif os.path.isdir(full_path):
            print(
                "文件夹：", f,
                "；内部示例：", os.listdir(full_path)[:3]
            )

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("/kaggle/input/datasets/dididoudou")
TRAIN_ROOT = BASE / "glaucoma-data"
VALID_ROOT = BASE / "glaucoma" / "valid"
TEST_ROOT = BASE / "glaucoma-firtst-data"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def count_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(
        1 for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

print("===== 1. 训练集图片 =====")
train_total = 0

for i in range(1, 6):
    folder = TRAIN_ROOT / f"train_{i}" / f"train_{i}"
    count = count_images(folder)
    train_total += count
    print(f"train_{i}: {count} 张 | 路径存在: {folder.exists()}")

print("训练集图片总数:", train_total)

print("\n===== 2. 验证集图片 =====")
print("路径:", VALID_ROOT)
print("路径存在:", VALID_ROOT.exists())
print("验证集图片数:", count_images(VALID_ROOT))

print("\n===== 3. 测试集图片 =====")
print("路径:", TEST_ROOT)
print("路径存在:", TEST_ROOT.exists())

if TEST_ROOT.exists():
    print("顶层内容:", list(TEST_ROOT.iterdir())[:10])
    print("测试集图片数:", count_images(TEST_ROOT))

print("\n===== 4. CSV 检查 =====")
for name in ["train.csv", "valid.csv", "test.csv"]:
    path = TRAIN_ROOT / name
    print(f"\n{name} | 存在: {path.exists()}")

    if path.exists():
        df = pd.read_csv(path)
        print("行数:", len(df))
        print("列名:", df.columns.tolist())
        print(df.head(2).to_string(index=False))

In [ ]:
import torch

print("PyTorch 版本：", torch.__version__)
print("GPU 是否可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 型号：", torch.cuda.get_device_name(0))
    print(
        "GPU 显存：",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("当前没有可用的 GPU，请检查 Notebook 设置中的 Accelerator。")

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image

# 数据集根目录
BASE = Path("/kaggle/input/datasets/dididoudou")

# 训练图片分布在五个子文件夹
TRAIN_DIRS = [
    BASE / "glaucoma-data" / f"train_{i}" / f"train_{i}"
    for i in range(1, 6)
]

# 验证图片和 CSV 的实际位置
VALID_DIR = BASE / "glaucoma" / "valid"
CSV_ROOT = BASE / "glaucoma-data"

train_df = pd.read_csv(CSV_ROOT / "train.csv")
valid_df = pd.read_csv(CSV_ROOT / "valid.csv")

print("===== CSV 检查 =====")
print("训练集行数：", len(train_df))
print("验证集行数：", len(valid_df))
print("训练集阳性数量：", int(train_df["True_Label"].sum()))
print("验证集阳性数量：", int(valid_df["True_Label"].sum()))

# 建立不区分文件名大小写的索引
# 这样可以处理 .jpg 和 .JPG 的差异
def build_image_index(directories):
    index = {}

    for directory in directories:
        directory = Path(directory)

        if not directory.exists():
            raise FileNotFoundError(f"目录不存在：{directory}")

        for path in directory.iterdir():
            if path.is_file() and path.suffix.lower() in {
                ".jpg", ".jpeg", ".png", ".bmp"
            }:
                key = path.name.lower()

                if key in index:
                    raise ValueError(f"发现重复图片名称：{path.name}")

                index[key] = path

    return index

train_index = build_image_index(TRAIN_DIRS)
valid_index = build_image_index([VALID_DIR])

print("\n===== 图片数量 =====")
print("训练图片：", len(train_index))
print("验证图片：", len(valid_index))

# 检查 CSV 中的名称是否都能找到
train_names = train_df["Image_name"].astype(str).str.lower()
valid_names = valid_df["Image_name"].astype(str).str.lower()

missing_train = [
    name for name in train_names if name not in train_index
]
missing_valid = [
    name for name in valid_names if name not in valid_index
]

print("\n===== 图片路径匹配 =====")
print("训练集找不到的图片：", len(missing_train))
print("验证集找不到的图片：", len(missing_valid))

if missing_train:
    print("训练集缺失示例：", missing_train[:5])

if missing_valid:
    print("验证集缺失示例：", missing_valid[:5])

# 实际读取图片，确保不是只有文件名存在
print("\n===== 实际读取测试 =====")

for name in train_names.head(3):
    with Image.open(train_index[name]) as img:
        img.verify()
    print("训练图片读取成功：", train_index[name].name)

for name in valid_names.head(3):
    with Image.open(valid_index[name]) as img:
        img.verify()
    print("验证图片读取成功：", valid_index[name].name)

if not missing_train and not missing_valid:
    print("\n初步数据检查通过，可以准备训练。")
else:
    print("\n数据检查发现缺失，请先不要开始训练。")

In [ ]:

%%writefile /kaggle/working/train_kaggle.py
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm


# ==================== 配置 ====================

BASE = Path("/kaggle/input/datasets/dididoudou")

TRAIN_ROOT = BASE / "glaucoma-data"
VALID_IMAGE_DIR = BASE / "glaucoma" / "valid"

TRAIN_CSV = TRAIN_ROOT / "train.csv"
VALID_CSV = TRAIN_ROOT / "valid.csv"

TRAIN_IMAGE_DIRS = [
    TRAIN_ROOT / f"train_{i}" / f"train_{i}"
    for i in range(1, 6)
]

IMAGE_SIZE = 384
BATCH_SIZE = 16
EPOCHS = 5
NUM_WORKERS = 2
LEARNING_RATE = 3e-4

OUTPUT_DIR = Path("/kaggle/working")
BEST_MODEL_PATH = OUTPUT_DIR / "glaucoma_efficientnet_b0_best.pth"
LOG_PATH = OUTPUT_DIR / "training_log.csv"

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True


# ==================== 路径与数据检查 ====================

def build_image_index(directories):
    """建立图片文件名到真实路径的映射，不区分扩展名大小写。"""
    index = {}

    for folder in directories:
        folder = Path(folder)

        if not folder.is_dir():
            raise FileNotFoundError(f"图片目录不存在：{folder}")

        count = 0

        for path in folder.iterdir():
            if not path.is_file():
                continue

            if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
                continue

            key = path.name.lower()

            if key in index:
                raise ValueError(f"发现重复图片文件名：{path.name}")

            index[key] = path
            count += 1

        print(f"图片目录：{folder.name}，图片数量：{count}")

    return index


def load_csvs():
    if not TRAIN_CSV.is_file():
        raise FileNotFoundError(f"找不到训练 CSV：{TRAIN_CSV}")

    if not VALID_CSV.is_file():
        raise FileNotFoundError(f"找不到验证 CSV：{VALID_CSV}")

    train_df = pd.read_csv(TRAIN_CSV)
    valid_df = pd.read_csv(VALID_CSV)

    required_train = {"Image_name", "Age", "True_Label"}
    required_valid = {"Image_name", "Age", "True_Label"}

    if not required_train.issubset(train_df.columns):
        raise ValueError(f"训练 CSV 列不符合预期：{train_df.columns.tolist()}")

    if not required_valid.issubset(valid_df.columns):
        raise ValueError(f"验证 CSV 列不符合预期：{valid_df.columns.tolist()}")

    for df in (train_df, valid_df):
        df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
        df["Image_name"] = df["Image_name"].astype(str)

    train_df["True_Label"] = pd.to_numeric(
        train_df["True_Label"], errors="raise"
    ).astype(int)

    valid_df["True_Label"] = pd.to_numeric(
        valid_df["True_Label"], errors="raise"
    ).astype(int)

    if not set(train_df["True_Label"].unique()).issubset({0, 1}):
        raise ValueError("训练标签必须为 0 或 1")

    if not set(valid_df["True_Label"].unique()).issubset({0, 1}):
        raise ValueError("验证标签必须为 0 或 1")

    # 年龄缺失值使用训练集年龄均值填补
    age_mean = float(train_df["Age"].mean())

    if not np.isfinite(age_mean):
        raise ValueError("训练集 Age 列没有可用的数值")

    age_std = float(train_df["Age"].std())

    if not np.isfinite(age_std) or age_std < 1e-6:
        age_std = 1.0

    train_df["Age"] = train_df["Age"].fillna(age_mean)
    valid_df["Age"] = valid_df["Age"].fillna(age_mean)

    return train_df, valid_df, age_mean, age_std


# ==================== 数据集 ====================

class GlaucomaDataset(Dataset):
    def __init__(self, dataframe, image_index, transform, age_mean, age_std):
        self.df = dataframe.reset_index(drop=True)
        self.image_index = image_index
        self.transform = transform
        self.age_mean = age_mean
        self.age_std = age_std

        names = self.df["Image_name"].astype(str).tolist()
        self.keys = [name.lower() for name in names]

        missing = [name for name, key in zip(names, self.keys)
                   if key not in image_index]

        if missing:
            raise FileNotFoundError(
                f"有 {len(missing)} 张图片未找到，示例：{missing[:5]}"
            )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        key = self.keys[idx]
        image_path = self.image_index[key]

        with Image.open(image_path) as img:
            image = img.convert("RGB")

        image = self.transform(image)

        age = (float(row["Age"]) - self.age_mean) / self.age_std
        age = torch.tensor(age, dtype=torch.float32)

        label = torch.tensor(float(row["True_Label"]), dtype=torch.float32)

        return image, age, label


# ==================== 模型 ====================

class EfficientNetB0WithAge(nn.Module):
    def __init__(self):
        super().__init__()

        # 不加载外部预训练权重，仅使用比赛训练集学习
        self.backbone = models.efficientnet_b0(weights=None)
        self.backbone.classifier = nn.Identity()

        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(1281, 1)
        )

    def forward(self, image, age):
        features = self.backbone(image)

        if age.ndim == 1:
            age = age.unsqueeze(1)

        combined = torch.cat([features, age], dim=1)
        return self.head(combined).squeeze(1)


# ==================== 验证 ====================

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    losses = []
    all_labels = []
    all_probs = []

    for images, ages, labels in tqdm(loader, desc="验证", leave=False):
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(images, ages)
            loss = criterion(logits, labels)

        probs = torch.sigmoid(logits.float())

        losses.append(loss.item())
        all_labels.extend(labels.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    y_true = np.asarray(all_labels)
    y_prob = np.asarray(all_probs)

    auc = roc_auc_score(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)

    return float(np.mean(losses)), float(auc), float(ap)


# ==================== 训练 ====================

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("=" * 60)
    print("设备：", device)

    if device.type == "cuda":
        print("GPU：", torch.cuda.get_device_name(0))
        print(
            "显存：",
            round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
            "GB"
        )

    train_df, valid_df, age_mean, age_std = load_csvs()

    print("\n训练 CSV 行数：", len(train_df))
    print("验证 CSV 行数：", len(valid_df))
    print("训练阳性：", int(train_df["True_Label"].sum()))
    print("验证阳性：", int(valid_df["True_Label"].sum()))
    print("年龄均值和标准差：", age_mean, age_std)

    print("\n建立图片索引……")
    train_index = build_image_index(TRAIN_IMAGE_DIRS)
    valid_index = build_image_index([VALID_IMAGE_DIR])

    print("\n训练图片总数：", len(train_index))
    print("验证图片总数：", len(valid_index))

    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.ColorJitter(
            brightness=0.08, contrast=0.08, saturation=0.05
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    valid_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    train_dataset = GlaucomaDataset(
        train_df, train_index, train_transform, age_mean, age_std
    )

    valid_dataset = GlaucomaDataset(
        valid_df, valid_index, valid_transform, age_mean, age_std
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0)
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0)
    )

    # CHECK_ONLY=1 时只读取小批量数据，不开始训练
    if os.environ.get("CHECK_ONLY", "0") == "1":
        images, ages, labels = next(iter(train_loader))
        print("\n训练 batch 图片形状：", tuple(images.shape))
        print("年龄形状：", tuple(ages.shape))
        print("标签形状：", tuple(labels.shape))
        print("batch 阳性数：", int(labels.sum().item()))

        images, ages, labels = next(iter(valid_loader))
        print("验证 batch 图片形状：", tuple(images.shape))
        print("\n数据加载测试通过，尚未开始训练。")
        return

    model = EfficientNetB0WithAge().to(device)

    pos_count = int(train_df["True_Label"].sum())
    neg_count = len(train_df) - pos_count
    pos_weight_value = neg_count / max(pos_count, 1)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight_value], device=device)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_auc = -1.0
    history = []

    print("\n开始训练，共", EPOCHS, "轮。")
    print("类别正样本权重：", round(pos_weight_value, 3))

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

        for images, ages, labels in progress:
            images = images.to(device, non_blocking=True)
            ages = ages.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(images, ages)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(loss.item())
            progress.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = float(np.mean(train_losses))
        valid_loss, valid_auc, valid_ap = evaluate(
            model, valid_loader, criterion, device
        )

        scheduler.step(valid_auc)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"\nEpoch {epoch}: "
            f"train_loss={train_loss:.4f}, "
            f"valid_loss={valid_loss:.4f}, "
            f"AUROC={valid_auc:.5f}, "
            f"AUPRC={valid_ap:.5f}, "
            f"lr={current_lr:.2e}"
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            "valid_auroc": valid_auc,
            "valid_auprc": valid_ap,
            "learning_rate": current_lr
        })

        pd.DataFrame(history).to_csv(LOG_PATH, index=False)

        if valid_auc > best_auc:
            best_auc = valid_auc

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "best_valid_auroc": valid_auc,
                "best_valid_auprc": valid_ap,
                "age_mean": age_mean,
                "age_std": age_std,
                "image_size": IMAGE_SIZE,
                "architecture": "efficientnet_b0_with_age"
            }, BEST_MODEL_PATH)

            print("已保存当前最佳模型：", BEST_MODEL_PATH)

    print("\n训练结束。")
    print("最佳验证 AUROC：", round(best_auc, 5))
    print("最佳模型：", BEST_MODEL_PATH)
    print("训练日志：", LOG_PATH)


if __name__ == "__main__":
    main()


In [ ]:
!CHECK_ONLY=1 python /kaggle/working/train_kaggle.py

In [ ]:
!python /kaggle/working/train_kaggle.py

In [ ]:
%%writefile /kaggle/working/train_kaggle.py
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm


# ==================== 配置 ====================

BASE = Path("/kaggle/input/datasets/dididoudou")

TRAIN_ROOT = BASE / "glaucoma-data"
VALID_IMAGE_DIR = BASE / "glaucoma" / "valid"

TRAIN_CSV = TRAIN_ROOT / "train.csv"
VALID_CSV = TRAIN_ROOT / "valid.csv"

TRAIN_IMAGE_DIRS = [
    TRAIN_ROOT / f"train_{i}" / f"train_{i}"
    for i in range(1, 6)
]

IMAGE_SIZE = 384
BATCH_SIZE = 16
EPOCHS = 5
NUM_WORKERS = 2
LEARNING_RATE = 3e-4

OUTPUT_DIR = Path("/kaggle/working")
BEST_MODEL_PATH = OUTPUT_DIR / "glaucoma_efficientnet_b0_best.pth"
LOG_PATH = OUTPUT_DIR / "training_log.csv"

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True


# ==================== 路径与数据检查 ====================

def build_image_index(directories):
    """建立图片文件名到真实路径的映射，不区分扩展名大小写。"""
    index = {}

    for folder in directories:
        folder = Path(folder)

        if not folder.is_dir():
            raise FileNotFoundError(f"图片目录不存在：{folder}")

        count = 0

        for path in folder.iterdir():
            if not path.is_file():
                continue

            if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
                continue

            key = path.name.lower()

            if key in index:
                raise ValueError(f"发现重复图片文件名：{path.name}")

            index[key] = path
            count += 1

        print(f"图片目录：{folder.name}，图片数量：{count}")

    return index


def load_csvs():
    if not TRAIN_CSV.is_file():
        raise FileNotFoundError(f"找不到训练 CSV：{TRAIN_CSV}")

    if not VALID_CSV.is_file():
        raise FileNotFoundError(f"找不到验证 CSV：{VALID_CSV}")

    train_df = pd.read_csv(TRAIN_CSV)
    valid_df = pd.read_csv(VALID_CSV)

    required_train = {"Image_name", "Age", "True_Label"}
    required_valid = {"Image_name", "Age", "True_Label"}

    if not required_train.issubset(train_df.columns):
        raise ValueError(f"训练 CSV 列不符合预期：{train_df.columns.tolist()}")

    if not required_valid.issubset(valid_df.columns):
        raise ValueError(f"验证 CSV 列不符合预期：{valid_df.columns.tolist()}")

    for df in (train_df, valid_df):
        df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
        df["Image_name"] = df["Image_name"].astype(str)

    train_df["True_Label"] = pd.to_numeric(
        train_df["True_Label"], errors="raise"
    ).astype(int)

    valid_df["True_Label"] = pd.to_numeric(
        valid_df["True_Label"], errors="raise"
    ).astype(int)

    if not set(train_df["True_Label"].unique()).issubset({0, 1}):
        raise ValueError("训练标签必须为 0 或 1")

    if not set(valid_df["True_Label"].unique()).issubset({0, 1}):
        raise ValueError("验证标签必须为 0 或 1")

    # 年龄缺失值使用训练集年龄均值填补
    age_mean = float(train_df["Age"].mean())

    if not np.isfinite(age_mean):
        raise ValueError("训练集 Age 列没有可用的数值")

    age_std = float(train_df["Age"].std())

    if not np.isfinite(age_std) or age_std < 1e-6:
        age_std = 1.0

    train_df["Age"] = train_df["Age"].fillna(age_mean)
    valid_df["Age"] = valid_df["Age"].fillna(age_mean)

    return train_df, valid_df, age_mean, age_std


# ==================== 数据集 ====================

class GlaucomaDataset(Dataset):
    def __init__(self, dataframe, image_index, transform, age_mean, age_std):
        self.df = dataframe.reset_index(drop=True)
        self.image_index = image_index
        self.transform = transform
        self.age_mean = age_mean
        self.age_std = age_std

        names = self.df["Image_name"].astype(str).tolist()
        self.keys = [name.lower() for name in names]

        missing = [name for name, key in zip(names, self.keys)
                   if key not in image_index]

        if missing:
            raise FileNotFoundError(
                f"有 {len(missing)} 张图片未找到，示例：{missing[:5]}"
            )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        key = self.keys[idx]
        image_path = self.image_index[key]

        with Image.open(image_path) as img:
            image = img.convert("RGB")

        image = self.transform(image)

        age = (float(row["Age"]) - self.age_mean) / self.age_std
        age = torch.tensor(age, dtype=torch.float32)

        label = torch.tensor(float(row["True_Label"]), dtype=torch.float32)

        return image, age, label


# ==================== 模型 ====================

class EfficientNetB0WithAge(nn.Module):
    def __init__(self):
        super().__init__()

        # 不加载外部预训练权重，仅使用比赛训练集学习
        self.backbone = models.efficientnet_b0(weights=None)
        self.backbone.classifier = nn.Identity()

        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(1281, 1)
        )

    def forward(self, image, age):
        features = self.backbone(image)

        if age.ndim == 1:
            age = age.unsqueeze(1)

        combined = torch.cat([features, age], dim=1)
        return self.head(combined).squeeze(1)


# ==================== 验证 ====================

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    losses = []
    all_labels = []
    all_probs = []

    for images, ages, labels in tqdm(loader, desc="验证", leave=False):
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(images, ages)
            loss = criterion(logits, labels)

        probs = torch.sigmoid(logits.float())

        losses.append(loss.item())
        all_labels.extend(labels.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())

    y_true = np.asarray(all_labels)
    y_prob = np.asarray(all_probs)

    auc = roc_auc_score(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)

    return float(np.mean(losses)), float(auc), float(ap)


# ==================== 训练 ====================

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("=" * 60)
    print("设备：", device)

    if device.type == "cuda":
        print("GPU：", torch.cuda.get_device_name(0))
        print(
            "显存：",
            round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
            "GB"
        )

    train_df, valid_df, age_mean, age_std = load_csvs()

    print("\n训练 CSV 行数：", len(train_df))
    print("验证 CSV 行数：", len(valid_df))
    print("训练阳性：", int(train_df["True_Label"].sum()))
    print("验证阳性：", int(valid_df["True_Label"].sum()))
    print("年龄均值和标准差：", age_mean, age_std)

    print("\n建立图片索引……")
    train_index = build_image_index(TRAIN_IMAGE_DIRS)
    valid_index = build_image_index([VALID_IMAGE_DIR])

    print("\n训练图片总数：", len(train_index))
    print("验证图片总数：", len(valid_index))

    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.ColorJitter(
            brightness=0.08, contrast=0.08, saturation=0.05
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    valid_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    train_dataset = GlaucomaDataset(
        train_df, train_index, train_transform, age_mean, age_std
    )

    valid_dataset = GlaucomaDataset(
        valid_df, valid_index, valid_transform, age_mean, age_std
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0)
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0)
    )

    # CHECK_ONLY=1 时只读取小批量数据，不开始训练
    if os.environ.get("CHECK_ONLY", "0") == "1":
        images, ages, labels = next(iter(train_loader))
        print("\n训练 batch 图片形状：", tuple(images.shape))
        print("年龄形状：", tuple(ages.shape))
        print("标签形状：", tuple(labels.shape))
        print("batch 阳性数：", int(labels.sum().item()))

        images, ages, labels = next(iter(valid_loader))
        print("验证 batch 图片形状：", tuple(images.shape))
        print("\n数据加载测试通过，尚未开始训练。")
        return

    model = EfficientNetB0WithAge().to(device)

    pos_count = int(train_df["True_Label"].sum())
    neg_count = len(train_df) - pos_count
    pos_weight_value = neg_count / max(pos_count, 1)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight_value], device=device)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_auc = -1.0
    history = []

    print("\n开始训练，共", EPOCHS, "轮。")
    print("类别正样本权重：", round(pos_weight_value, 3))

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

        for images, ages, labels in progress:
            images = images.to(device, non_blocking=True)
            ages = ages.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(images, ages)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_losses.append(loss.item())
            progress.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = float(np.mean(train_losses))
        valid_loss, valid_auc, valid_ap = evaluate(
            model, valid_loader, criterion, device
        )

        scheduler.step(valid_auc)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"\nEpoch {epoch}: "
            f"train_loss={train_loss:.4f}, "
            f"valid_loss={valid_loss:.4f}, "
            f"AUROC={valid_auc:.5f}, "
            f"AUPRC={valid_ap:.5f}, "
            f"lr={current_lr:.2e}"
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            "valid_auroc": valid_auc,
            "valid_auprc": valid_ap,
            "learning_rate": current_lr
        })

        pd.DataFrame(history).to_csv(LOG_PATH, index=False)

        if valid_auc > best_auc:
            best_auc = valid_auc

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "best_valid_auroc": valid_auc,
                "best_valid_auprc": valid_ap,
                "age_mean": age_mean,
                "age_std": age_std,
                "image_size": IMAGE_SIZE,
                "architecture": "efficientnet_b0_with_age"
            }, BEST_MODEL_PATH)

            print("已保存当前最佳模型：", BEST_MODEL_PATH)

    print("\n训练结束。")
    print("最佳验证 AUROC：", round(best_auc, 5))
    print("最佳模型：", BEST_MODEL_PATH)
    print("训练日志：", LOG_PATH)


if __name__ == "__main__":
    main()

In [ ]:
!python /kaggle/working/train_kaggle.py

In [ ]:
import os

print("当前目录：", os.getcwd())
print("working 目录存在：", os.path.exists("/kaggle/working"))

if os.path.exists("/kaggle/working"):
    print("working 目录文件：", os.listdir("/kaggle/working"))

print("\n查找训练脚本：")
for root, dirs, files in os.walk("/kaggle/working"):
    for name in files:
        if "train_kaggle" in name:
            print(os.path.join(root, name))

In [ ]:
import os

script = "/kaggle/working/train_kaggle.py"

print("脚本是否存在：", os.path.exists(script))

if os.path.exists(script):
    print("脚本大小：", os.path.getsize(script), "bytes")
    print("可以继续进行数据加载检查。")
else:
    print("脚本仍然不存在，请勿开始训练。")

In [ ]:
!CHECK_ONLY=1 python /kaggle/working/train_kaggle.py

In [ ]:
!python /kaggle/working/train_kaggle.py

In [ ]:
!nvidia-smi

In [ ]:

%%writefile /kaggle/working/train_kaggle.py
import os
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==================== 配置 ====================
DATA_ROOT = Path("/kaggle/input/datasets/dididoudou")
WORK = Path("/kaggle/working")

TRAIN_CSV = DATA_ROOT / "glaucoma-data" / "train.csv"
VALID_CSV = DATA_ROOT / "glaucoma-data" / "valid.csv"
TEST_CSV  = DATA_ROOT / "glaucoma-data" / "test.csv"

IMAGE_SIZE = 320
BATCH_SIZE = 32
EPOCHS = 2
NUM_WORKERS = 2
LR = 3e-4
SEED = 42

BEST_MODEL = WORK / "glaucoma_best.pth"
LOG_FILE = WORK / "training_log.csv"
SUBMISSION_FILE = WORK / "submission.csv"

# ==================== 初始化 ====================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60, flush=True)
print("设备：", device, flush=True)
if device.type == "cuda":
    print("GPU：", torch.cuda.get_device_name(0), flush=True)
    print("显存：", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB", flush=True)

for p in [TRAIN_CSV, VALID_CSV, TEST_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"找不到 CSV：{p}")

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"训练集：{len(train_df)} 张；阳性：{int(train_df['True_Label'].sum())}", flush=True)
print(f"验证集：{len(valid_df)} 张；阳性：{int(valid_df['True_Label'].sum())}", flush=True)
print(f"测试 CSV：{len(test_df)} 张", flush=True)

# ==================== 建立图片索引 ====================
# 在已挂载的数据集内查找图片，避免假定测试图片所在目录。
print("\n正在扫描已挂载数据集中的图片……", flush=True)
image_index = {}
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

for root, dirs, files in os.walk("/kaggle/input"):
    for name in files:
        if Path(name).suffix.lower() in image_exts:
            # 如果重名，保留第一个路径
            image_index.setdefault(name.lower(), os.path.join(root, name))

print("找到图片文件：", len(image_index), flush=True)

def find_image(name):
    return image_index.get(str(name).lower())

train_df["_path"] = train_df["Image_name"].map(find_image)
valid_df["_path"] = valid_df["Image_name"].map(find_image)
test_df["_path"] = test_df["Image_name"].map(find_image)

print("训练图片匹配：", int(train_df["_path"].notna().sum()), "/", len(train_df), flush=True)
print("验证图片匹配：", int(valid_df["_path"].notna().sum()), "/", len(valid_df), flush=True)
print("测试图片匹配：", int(test_df["_path"].notna().sum()), "/", len(test_df), flush=True)

if train_df["_path"].isna().any() or valid_df["_path"].isna().any():
    raise RuntimeError("训练集或验证集存在缺失图片，请检查数据集挂载。")

if test_df["_path"].isna().any():
    missing = test_df.loc[test_df["_path"].isna(), "Image_name"].head(10).tolist()
    raise RuntimeError(
        "测试图片没有全部找到，因此暂时不能生成有效的比赛预测 CSV。\n"
        f"缺失示例：{missing}\n"
        "请在 Notebook 右侧 Add Input 中添加包含 test 图片的数据集后再运行。"
    )

# ==================== 数据集 ====================
train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

age_mean = float(train_df["Age"].mean())
age_std = max(float(train_df["Age"].std()), 1.0)

class FundusDataset(Dataset):
    def __init__(self, df, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["_path"]) as im:
            image = im.convert("RGB")
            image = self.transform(image)

        age = float(row["Age"])
        age = (age - age_mean) / age_std

        if self.is_test:
            return image, torch.tensor(age, dtype=torch.float32)

        label = float(row["True_Label"])
        return image, torch.tensor(age, dtype=torch.float32), torch.tensor(label)

train_ds = FundusDataset(train_df, train_tf)
valid_ds = FundusDataset(valid_df, eval_tf)
test_ds = FundusDataset(test_df, eval_tf, is_test=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)

# ==================== 模型 ====================
# 仅使用 torchvision 自带结构，不下载外部预训练权重。
model = models.efficientnet_b0(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features + 1, 1)
model = model.to(device)

pos = max(int(train_df["True_Label"].sum()), 1)
neg = len(train_df) - pos
pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def forward_model(images, ages):
    features = model.features(images)
    features = model.avgpool(features)
    features = torch.flatten(features, 1)
    combined = torch.cat([features, ages.unsqueeze(1)], dim=1)
    return model.classifier(combined).squeeze(1)

# ==================== 训练和验证 ====================
history = []
best_auc = -1.0

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    model.train()
    running_loss = 0.0
    seen = 0

    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========", flush=True)
    pbar = tqdm(train_loader, desc=f"训练 {epoch}/{EPOCHS}", dynamic_ncols=True)

    for step, (images, ages, labels) in enumerate(pbar, start=1):
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=use_amp
        ):
            logits = forward_model(images, ages)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = labels.size(0)
        running_loss += loss.item() * bs
        seen += bs

        pbar.set_postfix(
            loss=f"{running_loss / max(seen, 1):.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}"
        )

    train_loss = running_loss / max(seen, 1)
    model.eval()
    val_probs, val_labels = [], []
    val_loss_sum, val_seen = 0.0, 0

    with torch.inference_mode():
        pbar = tqdm(valid_loader, desc="验证集", dynamic_ncols=True)
        for images, ages, labels in pbar:
            images = images.to(device, non_blocking=True)
            ages = ages.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float()

            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=use_amp
            ):
                logits = forward_model(images, ages)
                loss = criterion(logits, labels)

            probs = torch.sigmoid(logits.float())
            val_probs.extend(probs.cpu().numpy().tolist())
            val_labels.extend(labels.cpu().numpy().tolist())
            val_loss_sum += loss.item() * labels.size(0)
            val_seen += labels.size(0)

    val_loss = val_loss_sum / max(val_seen, 1)
    try:
        val_auc = roc_auc_score(val_labels, val_probs)
        val_auprc = average_precision_score(val_labels, val_probs)
    except ValueError:
        val_auc, val_auprc = float("nan"), float("nan")

    elapsed = time.time() - start
    print(
        f"\nEpoch {epoch}/{EPOCHS} 完成 | "
        f"Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | "
        f"AUROC={val_auc:.5f} | AUPRC={val_auprc:.5f} | "
        f"耗时={elapsed/60:.1f} 分钟",
        flush=True
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": val_loss,
        "AUROC": val_auc,
        "AUPRC": val_auprc,
        "minutes": elapsed / 60
    })
    pd.DataFrame(history).to_csv(LOG_FILE, index=False)

    if np.isfinite(val_auc) and val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            "model_state_dict": model.state_dict(),
            "age_mean": age_mean,
            "age_std": age_std,
            "image_size": IMAGE_SIZE,
            "epoch": epoch,
            "valid_auc": val_auc,
            "valid_auprc": val_auprc
        }, BEST_MODEL)
        print("已保存当前最佳模型：", BEST_MODEL, flush=True)

    scheduler.step()

# ==================== 加载最佳模型并预测测试集 ====================
if BEST_MODEL.exists():
    checkpoint = torch.load(BEST_MODEL, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(
        f"\n使用第 {checkpoint['epoch']} 轮最佳模型预测；"
        f"验证 AUROC={checkpoint['valid_auc']:.5f}",
        flush=True
    )

model.eval()
predictions = []

with torch.inference_mode():
    pbar = tqdm(test_loader, desc="生成测试集预测", dynamic_ncols=True)
    for images, ages in pbar:
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=use_amp
        ):
            logits = forward_model(images, ages)

        predictions.extend(torch.sigmoid(logits.float()).cpu().numpy().tolist())

# True_Label 列填写阳性概率，保留连续分数，适合 AUROC 类指标。
submission = pd.DataFrame({
    "Image_name": test_df["Image_name"],
    "True_Label": predictions
})
submission.to_csv(SUBMISSION_FILE, index=False)

print("\n全部完成！", flush=True)
print("验证结果日志：", LOG_FILE, flush=True)
print("最佳模型：", BEST_MODEL, flush=True)
print("比赛预测 CSV：", SUBMISSION_FILE, flush=True)
print(submission.head(), flush=True)

In [ ]:
!python -u /kaggle/working/train_kaggle.py

In [ ]:
 %%writefile /kaggle/working/train_kaggle.py
print("脚本创建成功！")
print("下一步可以检查文件是否存在。")

In [ ]:
import os

path = "/kaggle/working/train_kaggle.py"

print("文件存在：", os.path.exists(path))
print("文件大小：", os.path.getsize(path) if os.path.exists(path) else 0)

In [ ]:

%%writefile /kaggle/working/train_kaggle.py
import os
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==================== 配置 ====================
DATA_ROOT = Path("/kaggle/input/datasets/dididoudou")
WORK = Path("/kaggle/working")

TRAIN_CSV = DATA_ROOT / "glaucoma-data" / "train.csv"
VALID_CSV = DATA_ROOT / "glaucoma-data" / "valid.csv"
TEST_CSV  = DATA_ROOT / "glaucoma-data" / "test.csv"

IMAGE_SIZE = 320
BATCH_SIZE = 32
EPOCHS = 2
NUM_WORKERS = 2
LR = 3e-4
SEED = 42

BEST_MODEL = WORK / "glaucoma_best.pth"
LOG_FILE = WORK / "training_log.csv"
SUBMISSION_FILE = WORK / "submission.csv"

# ==================== 初始化 ====================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60, flush=True)
print("设备：", device, flush=True)
if device.type == "cuda":
    print("GPU：", torch.cuda.get_device_name(0), flush=True)
    print("显存：", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB", flush=True)

for p in [TRAIN_CSV, VALID_CSV, TEST_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"找不到 CSV：{p}")

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"训练集：{len(train_df)} 张；阳性：{int(train_df['True_Label'].sum())}", flush=True)
print(f"验证集：{len(valid_df)} 张；阳性：{int(valid_df['True_Label'].sum())}", flush=True)
print(f"测试 CSV：{len(test_df)} 张", flush=True)

# ==================== 建立图片索引 ====================
# 在已挂载的数据集内查找图片，避免假定测试图片所在目录。
print("\n正在扫描已挂载数据集中的图片……", flush=True)
image_index = {}
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

for root, dirs, files in os.walk("/kaggle/input"):
    for name in files:
        if Path(name).suffix.lower() in image_exts:
            # 如果重名，保留第一个路径
            image_index.setdefault(name.lower(), os.path.join(root, name))

print("找到图片文件：", len(image_index), flush=True)

def find_image(name):
    return image_index.get(str(name).lower())

train_df["_path"] = train_df["Image_name"].map(find_image)
valid_df["_path"] = valid_df["Image_name"].map(find_image)
test_df["_path"] = test_df["Image_name"].map(find_image)

print("训练图片匹配：", int(train_df["_path"].notna().sum()), "/", len(train_df), flush=True)
print("验证图片匹配：", int(valid_df["_path"].notna().sum()), "/", len(valid_df), flush=True)
print("测试图片匹配：", int(test_df["_path"].notna().sum()), "/", len(test_df), flush=True)

if train_df["_path"].isna().any() or valid_df["_path"].isna().any():
    raise RuntimeError("训练集或验证集存在缺失图片，请检查数据集挂载。")

if test_df["_path"].isna().any():
    missing = test_df.loc[test_df["_path"].isna(), "Image_name"].head(10).tolist()
    raise RuntimeError(
        "测试图片没有全部找到，因此暂时不能生成有效的比赛预测 CSV。\n"
        f"缺失示例：{missing}\n"
        "请在 Notebook 右侧 Add Input 中添加包含 test 图片的数据集后再运行。"
    )

# ==================== 数据集 ====================
train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

age_mean = float(train_df["Age"].mean())
age_std = max(float(train_df["Age"].std()), 1.0)

class FundusDataset(Dataset):
    def __init__(self, df, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["_path"]) as im:
            image = im.convert("RGB")
            image = self.transform(image)

        age = float(row["Age"])
        age = (age - age_mean) / age_std

        if self.is_test:
            return image, torch.tensor(age, dtype=torch.float32)

        label = float(row["True_Label"])
        return image, torch.tensor(age, dtype=torch.float32), torch.tensor(label)

train_ds = FundusDataset(train_df, train_tf)
valid_ds = FundusDataset(valid_df, eval_tf)
test_ds = FundusDataset(test_df, eval_tf, is_test=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)

# ==================== 模型 ====================
# 仅使用 torchvision 自带结构，不下载外部预训练权重。
model = models.efficientnet_b0(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features + 1, 1)
model = model.to(device)

pos = max(int(train_df["True_Label"].sum()), 1)
neg = len(train_df) - pos
pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def forward_model(images, ages):
    features = model.features(images)
    features = model.avgpool(features)
    features = torch.flatten(features, 1)
    combined = torch.cat([features, ages.unsqueeze(1)], dim=1)
    return model.classifier(combined).squeeze(1)

# ==================== 训练和验证 ====================
history = []
best_auc = -1.0

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    model.train()
    running_loss = 0.0
    seen = 0

    print(f"\n========== Epoch {epoch}/{EPOCHS} ==========", flush=True)
    pbar = tqdm(train_loader, desc=f"训练 {epoch}/{EPOCHS}", dynamic_ncols=True)

    for step, (images, ages, labels) in enumerate(pbar, start=1):
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=use_amp
        ):
            logits = forward_model(images, ages)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = labels.size(0)
        running_loss += loss.item() * bs
        seen += bs

        pbar.set_postfix(
            loss=f"{running_loss / max(seen, 1):.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}"
        )

    train_loss = running_loss / max(seen, 1)
    model.eval()
    val_probs, val_labels = [], []
    val_loss_sum, val_seen = 0.0, 0

    with torch.inference_mode():
        pbar = tqdm(valid_loader, desc="验证集", dynamic_ncols=True)
        for images, ages, labels in pbar:
            images = images.to(device, non_blocking=True)
            ages = ages.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float()

            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=use_amp
            ):
                logits = forward_model(images, ages)
                loss = criterion(logits, labels)

            probs = torch.sigmoid(logits.float())
            val_probs.extend(probs.cpu().numpy().tolist())
            val_labels.extend(labels.cpu().numpy().tolist())
            val_loss_sum += loss.item() * labels.size(0)
            val_seen += labels.size(0)

    val_loss = val_loss_sum / max(val_seen, 1)
    try:
        val_auc = roc_auc_score(val_labels, val_probs)
        val_auprc = average_precision_score(val_labels, val_probs)
    except ValueError:
        val_auc, val_auprc = float("nan"), float("nan")

    elapsed = time.time() - start
    print(
        f"\nEpoch {epoch}/{EPOCHS} 完成 | "
        f"Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | "
        f"AUROC={val_auc:.5f} | AUPRC={val_auprc:.5f} | "
        f"耗时={elapsed/60:.1f} 分钟",
        flush=True
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": val_loss,
        "AUROC": val_auc,
        "AUPRC": val_auprc,
        "minutes": elapsed / 60
    })
    pd.DataFrame(history).to_csv(LOG_FILE, index=False)

    if np.isfinite(val_auc) and val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            "model_state_dict": model.state_dict(),
            "age_mean": age_mean,
            "age_std": age_std,
            "image_size": IMAGE_SIZE,
            "epoch": epoch,
            "valid_auc": val_auc,
            "valid_auprc": val_auprc
        }, BEST_MODEL)
        print("已保存当前最佳模型：", BEST_MODEL, flush=True)

    scheduler.step()

# ==================== 加载最佳模型并预测测试集 ====================
if BEST_MODEL.exists():
    checkpoint = torch.load(BEST_MODEL, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(
        f"\n使用第 {checkpoint['epoch']} 轮最佳模型预测；"
        f"验证 AUROC={checkpoint['valid_auc']:.5f}",
        flush=True
    )

model.eval()
predictions = []

with torch.inference_mode():
    pbar = tqdm(test_loader, desc="生成测试集预测", dynamic_ncols=True)
    for images, ages in pbar:
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=use_amp
        ):
            logits = forward_model(images, ages)

        predictions.extend(torch.sigmoid(logits.float()).cpu().numpy().tolist())

# True_Label 列填写阳性概率，保留连续分数，适合 AUROC 类指标。
submission = pd.DataFrame({
    "Image_name": test_df["Image_name"],
    "True_Label": predictions
})
submission.to_csv(SUBMISSION_FILE, index=False)

print("\n全部完成！", flush=True)
print("验证结果日志：", LOG_FILE, flush=True)
print("最佳模型：", BEST_MODEL, flush=True)
print("比赛预测 CSV：", SUBMISSION_FILE, flush=True)
print(submission.head(), flush=True)

In [ ]:
import os

p = "/kaggle/working/train_kaggle.py"

print("文件存在：", os.path.exists(p))

if os.path.exists(p):
    print("文件大小：", os.path.getsize(p), "bytes")

In [ ]:

# ============================================================
# Glaucoma Competition Baseline
# 实时进度 + 验证 AUROC/AUPRC + 保存模型 + 生成预测 CSV
# 直接在 Kaggle Notebook 的一个代码单元格中运行
# ============================================================

import os
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# -------------------- 1. 配置 --------------------
ROOT = Path("/kaggle/input/datasets/dididoudou/glaucoma-data")
WORK = Path("/kaggle/working")

TRAIN_CSV = ROOT / "train.csv"
VALID_CSV = ROOT / "valid.csv"
TEST_CSV = ROOT / "test.csv"

IMAGE_SIZE = 320
BATCH_SIZE = 32
EPOCHS = 2
NUM_WORKERS = 2
LR = 3e-4
SEED = 42

BEST_PATH = WORK / "glaucoma_best.pth"
LOG_PATH = WORK / "training_log.csv"
SUBMISSION_PATH = WORK / "submission.csv"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

print("=" * 60, flush=True)
print("设备：", device, flush=True)
if use_amp:
    print("GPU：", torch.cuda.get_device_name(0), flush=True)
    print("显存：",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB", flush=True)

# -------------------- 2. 检查 CSV --------------------
for p in [TRAIN_CSV, VALID_CSV, TEST_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"找不到文件：{p}\n请检查 Kaggle 右侧 Add Input 数据集。"
        )

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)
test_df = pd.read_csv(TEST_CSV)

required_train = {"Image_name", "Age", "True_Label"}
required_test = {"Image_name", "Age"}

if not required_train.issubset(train_df.columns):
    raise ValueError(f"训练 CSV 列不符合预期：{train_df.columns.tolist()}")
if not required_train.issubset(valid_df.columns):
    raise ValueError(f"验证 CSV 列不符合预期：{valid_df.columns.tolist()}")
if not required_test.issubset(test_df.columns):
    raise ValueError(f"测试 CSV 列不符合预期：{test_df.columns.tolist()}")

print(f"\n训练集：{len(train_df)}，阳性：{int(train_df['True_Label'].sum())}",
      flush=True)
print(f"验证集：{len(valid_df)}，阳性：{int(valid_df['True_Label'].sum())}",
      flush=True)
print(f"测试集 CSV：{len(test_df)}", flush=True)

# -------------------- 3. 建立图片索引 --------------------
# 图片在 Kaggle 已挂载的数据集中；只扫描图片文件，不读取整张图片内容。
print("\n正在查找图片文件，请稍候……", flush=True)
image_index = {}

for folder, dirs, files in os.walk("/kaggle/input"):
    for name in files:
        if Path(name).suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"
        }:
            image_index.setdefault(name.lower(), os.path.join(folder, name))

print("图片索引数量：", len(image_index), flush=True)

def lookup_image(name):
    return image_index.get(str(name).lower())

train_df["_path"] = train_df["Image_name"].map(lookup_image)
valid_df["_path"] = valid_df["Image_name"].map(lookup_image)
test_df["_path"] = test_df["Image_name"].map(lookup_image)

print("训练图片匹配：", train_df["_path"].notna().sum(), "/", len(train_df),
      flush=True)
print("验证图片匹配：", valid_df["_path"].notna().sum(), "/", len(valid_df),
      flush=True)
print("测试图片匹配：", test_df["_path"].notna().sum(), "/", len(test_df),
      flush=True)

if train_df["_path"].isna().any() or valid_df["_path"].isna().any():
    raise RuntimeError("训练或验证图片缺失，暂时停止，避免训练错误。")

if test_df["_path"].isna().any():
    missing = test_df.loc[test_df["_path"].isna(), "Image_name"].head(10).tolist()
    raise RuntimeError(
        "测试图片未全部找到，现阶段不能生成完整预测 CSV。\n"
        f"缺失文件示例：{missing}\n"
        "请先通过右侧 Add Input 添加包含测试图片的数据集，再重新运行。"
    )

# -------------------- 4. 图片预处理 --------------------
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ColorJitter(
        brightness=0.08, contrast=0.08, saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    ),
])

age_mean = float(train_df["Age"].mean())
age_std = max(float(train_df["Age"].std()), 1.0)

class FundusDataset(Dataset):
    def __init__(self, df, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["_path"]) as img:
            img = img.convert("RGB")
            img = self.transform(img)

        age = (float(row["Age"]) - age_mean) / age_std

        if self.is_test:
            return img, torch.tensor(age, dtype=torch.float32)

        label = float(row["True_Label"])
        return img, torch.tensor(age, dtype=torch.float32), torch.tensor(label)

train_loader = DataLoader(
    FundusDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=use_amp,
    persistent_workers=(NUM_WORKERS > 0)
)

valid_loader = DataLoader(
    FundusDataset(valid_df, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=use_amp,
    persistent_workers=(NUM_WORKERS > 0)
)

test_loader = DataLoader(
    FundusDataset(test_df, eval_transform, is_test=True),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=use_amp,
    persistent_workers=(NUM_WORKERS > 0)
)

print("\nDataLoader 准备完成。", flush=True)
print("训练 batch 数：", len(train_loader), flush=True)
print("验证 batch 数：", len(valid_loader), flush=True)
print("测试 batch 数：", len(test_loader), flush=True)

# -------------------- 5. 建立模型 --------------------
# 不下载外部预训练权重。
model = models.efficientnet_b0(weights=None)
feature_count = model.classifier[1].in_features
model.classifier[1] = nn.Linear(feature_count + 1, 1)
model = model.to(device)

positive = max(int(train_df["True_Label"].sum()), 1)
negative = len(train_df) - positive
pos_weight = torch.tensor(
    [negative / positive], dtype=torch.float32, device=device
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def forward_model(images, ages):
    features = model.features(images)
    features = model.avgpool(features)
    features = torch.flatten(features, 1)
    features = torch.cat([features, ages.unsqueeze(1)], dim=1)
    return model.classifier(features).squeeze(1)

# -------------------- 6. 训练与验证 --------------------
history = []
best_auc = -1.0

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    total_loss = 0.0
    total_count = 0

    print(f"\n{'=' * 20} Epoch {epoch}/{EPOCHS} {'=' * 20}",
          flush=True)

    progress = tqdm(
        train_loader,
        desc=f"训练 {epoch}/{EPOCHS}",
        dynamic_ncols=True
    )

    for step, (images, ages, labels) in enumerate(progress, 1):
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float()

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):
            logits = forward_model(images, ages)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        n = labels.size(0)
        total_loss += loss.item() * n
        total_count += n

        progress.set_postfix(
            loss=f"{total_loss / total_count:.4f}",
            batch=f"{step}/{len(train_loader)}"
        )

    train_loss = total_loss / max(total_count, 1)

    model.eval()
    val_probs = []
    val_labels = []
    val_loss_sum = 0.0
    val_count = 0

    with torch.inference_mode():
        progress = tqdm(
            valid_loader,
            desc="验证集",
            dynamic_ncols=True
        )

        for images, ages, labels in progress:
            images = images.to(device, non_blocking=True)
            ages = ages.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float()

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):
                logits = forward_model(images, ages)
                loss = criterion(logits, labels)

            probs = torch.sigmoid(logits.float())
            val_probs.extend(probs.cpu().numpy().tolist())
            val_labels.extend(labels.cpu().numpy().tolist())
            val_loss_sum += loss.item() * labels.size(0)
            val_count += labels.size(0)

    val_loss = val_loss_sum / max(val_count, 1)
    val_auc = roc_auc_score(val_labels, val_probs)
    val_auprc = average_precision_score(val_labels, val_probs)
    elapsed = (time.time() - epoch_start) / 60

    print(
        f"\nEpoch {epoch}/{EPOCHS} 完成 | "
        f"Train Loss={train_loss:.4f} | "
        f"Val Loss={val_loss:.4f} | "
        f"AUROC={val_auc:.5f} | "
        f"AUPRC={val_auprc:.5f} | "
        f"耗时={elapsed:.1f} 分钟",
        flush=True
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": val_loss,
        "AUROC": val_auc,
        "AUPRC": val_auprc,
        "minutes": elapsed
    })
    pd.DataFrame(history).to_csv(LOG_PATH, index=False)

    if np.isfinite(val_auc) and val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "valid_auc": float(val_auc),
            "valid_auprc": float(val_auprc),
            "age_mean": age_mean,
            "age_std": age_std,
            "image_size": IMAGE_SIZE
        }, BEST_PATH)
        print("最佳模型已保存：", BEST_PATH, flush=True)

    scheduler.step()

# -------------------- 7. 使用最佳模型预测测试集 --------------------
checkpoint = torch.load(
    BEST_PATH, map_location=device, weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

predictions = []

with torch.inference_mode():
    progress = tqdm(
        test_loader,
        desc="预测测试集",
        dynamic_ncols=True
    )
    for images, ages in progress:
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):
            logits = forward_model(images, ages)

        predictions.extend(
            torch.sigmoid(logits.float()).cpu().numpy().tolist()
        )

# 先输出常见格式；提交前仍应对照比赛官方样例核实列名。
submission = pd.DataFrame({
    "Image_name": test_df["Image_name"],
    "True_Label": predictions
})
submission.to_csv(SUBMISSION_PATH, index=False)

print("\n" + "=" * 60, flush=True)
print("训练和预测已完成！", flush=True)
print("最佳验证 AUROC：", checkpoint["valid_auc"], flush=True)
print("最佳验证 AUPRC：", checkpoint["valid_auprc"], flush=True)
print("训练日志：", LOG_PATH, flush=True)
print("最佳模型：", BEST_PATH, flush=True)
print("预测 CSV：", SUBMISSION_PATH, flush=True)
print("CSV 行数：", len(submission), flush=True)
print(submission.head(), flush=True)

In [ ]:
import os
p = "/kaggle/working/train_kaggle.py"
print(os.path.exists(p), os.path.getsize(p) if os.path.exists(p) else 0)

In [ ]:
import os

print("正在查找 test_data 中的测试图片……")

for root, dirs, files in os.walk("/kaggle/input"):
    if "test_data" in root.lower():
        imgs = [
            f for f in files
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]
        if imgs:
            print("\n图片目录：", root)
            print("图片数量：", len(imgs))
            print("前5张：", imgs[:5])

In [ ]:
import shutil
from pathlib import Path
import os

WORK = Path("/kaggle/working")
backup_dir = WORK / "checkpoints"
backup_dir.mkdir(exist_ok=True)

for name in ["glaucoma_last.pth", "glaucoma_best.pth", "training_log.csv",
             "image_index_cache.pkl"]:
    src = WORK / name
    if src.exists():
        dst = backup_dir / name
        shutil.copy(src, dst)
        print(f"backed up: {name} ({src.stat().st_size/1024/1024:.1f} MB)")
    else:
        print(f"not found yet: {name}")

In [ ]:
# ============================================================
# Glaucoma Competition - Stable Full Version
# 稳定优先：NUM_WORKERS=0 | IMAGE_SIZE=320 | 无channels_last
# 保留：预训练 | warmup+cosine | 早停 | TTA(可选) | 断点续训 | 坏图容错
# ============================================================

import os
import gc
import time
import random
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# -------------------- 1. Configuration --------------------
DATA_ROOT = Path("/kaggle/input/datasets/dididoudou/glaucoma-data")
WORK = Path("/kaggle/working")

TRAIN_CSV = DATA_ROOT / "train.csv"
VALID_CSV = DATA_ROOT / "valid.csv"
TEST_CSV = DATA_ROOT / "test.csv"

# 可选：本地预训练权重路径（正式环境无网时用）
LOCAL_PRETRAINED = None   # 例如 Path("/kaggle/input/efficientnet-b0/efficientnet_b0.pth")

# ---- 稳定优先配置 ----
IMAGE_SIZE = 320          # 稳定后再试 384
BATCH_SIZE = 32           # 320 分辨率下 32 较稳；384 时改 16
EPOCHS = 20
WARMUP_EPOCHS = 1
PATIENCE = 5
NUM_WORKERS = 0           # 关键：先设 0，跑通后再试 2
LR_BACKBONE = 3e-5
LR_HEAD = 3e-4
SEED = 42

USE_TTA = True            # 水平翻转 TTA，显存不够时关掉

BEST_PATH = WORK / "glaucoma_best.pth"
LAST_PATH = WORK / "glaucoma_last.pth"
LOG_PATH = WORK / "training_log.csv"
SUBMISSION_PATH = WORK / "submission.csv"
INDEX_CACHE = WORK / "image_index_cache.pkl"

RESUME = True

# ---- 随机种子 ----
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("=" * 65, flush=True)
print("Device:", device, flush=True)
if use_amp:
    print("GPU数量：", torch.cuda.device_count(), flush=True)
    print("主GPU：", torch.cuda.get_device_name(0), flush=True)
    print("单卡显存：",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB", flush=True)
print(f"Image size: {IMAGE_SIZE} | Batch size: {BATCH_SIZE} | "
      f"Epochs: {EPOCHS} | Workers: {NUM_WORKERS} | TTA: {USE_TTA}", flush=True)

# -------------------- 2. Read CSV files --------------------
print("[STEP 1] 读取 CSV", flush=True)
for p in [TRAIN_CSV, VALID_CSV, TEST_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"CSV not found: {p}")

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)
test_df = pd.read_csv(TEST_CSV)

for name, df in [("train", train_df), ("valid", valid_df)]:
    needed = {"Image_name", "Age", "True_Label"}
    if not needed.issubset(df.columns):
        raise ValueError(f"{name}.csv columns unexpected: {df.columns.tolist()}")

if not {"Image_name", "Age"}.issubset(test_df.columns):
    raise ValueError(f"test.csv columns unexpected: {test_df.columns.tolist()}")

train_df["True_Label"] = train_df["True_Label"].astype(int)
valid_df["True_Label"] = valid_df["True_Label"].astype(int)

# 年龄缺失值填充
age_median = float(train_df["Age"].median())
print("Age median (from train):", age_median, flush=True)
for df in [train_df, valid_df, test_df]:
    df["Age"] = df["Age"].fillna(age_median)

print(f"Train: {len(train_df)}, positives: {int(train_df['True_Label'].sum())}",
      flush=True)
print(f"Valid: {len(valid_df)}, positives: {int(valid_df['True_Label'].sum())}",
      flush=True)
print(f"Test: {len(test_df)}", flush=True)

gc.collect()

# -------------------- 3. 图片索引（带缓存） --------------------
print("\n[STEP 2] 扫描/加载图片索引", flush=True)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

if INDEX_CACHE.exists():
    print("Load image index from cache...", flush=True)
    try:
        with open(INDEX_CACHE, "rb") as f:
            image_index = pickle.load(f)
        print("Cache loaded, images:", len(image_index), flush=True)
    except Exception as e:
        print("Cache broken, rebuild:", e, flush=True)
        INDEX_CACHE.unlink()
        image_index = None
else:
    image_index = None

if image_index is None:
    all_image_paths = []
    scan_root = "/kaggle/input/datasets/dididoudou"
    for root, dirs, files in os.walk(scan_root):
        for filename in files:
            if Path(filename).suffix.lower() in IMAGE_EXTS:
                all_image_paths.append(os.path.join(root, filename))

    print("Image files found:", len(all_image_paths), flush=True)
    image_index = {}
    for path in tqdm(all_image_paths, desc="Indexing images", dynamic_ncols=True):
        key = os.path.basename(path).lower()
        image_index.setdefault(key, path)

    with open(INDEX_CACHE, "wb") as f:
        pickle.dump(image_index, f)
    print("Index saved to cache.", flush=True)

print("Total images in index:", len(image_index), flush=True)


def find_image(filename):
    return image_index.get(os.path.basename(str(filename)).lower())


train_df["_path"] = train_df["Image_name"].map(find_image)
valid_df["_path"] = valid_df["Image_name"].map(find_image)
test_df["_path"] = test_df["Image_name"].map(find_image)

for name, df in [("Train", train_df), ("Valid", valid_df), ("Test", test_df)]:
    found = int(df["_path"].notna().sum())
    print(f"{name} image match: {found}/{len(df)}", flush=True)
    if found != len(df):
        missing = df.loc[df["_path"].isna(), "Image_name"].head(10).tolist()
        raise RuntimeError(
            f"{name} images missing. Examples: {missing}. "
            "Check that the correct image datasets are attached via Add Input."
        )

gc.collect()

# -------------------- 4. 数据集 + 预处理 --------------------
print("\n[STEP 3] 构造 Dataset / DataLoader", flush=True)

age_mean = float(train_df["Age"].mean())
age_std = max(float(train_df["Age"].std()), 1.0)
print(f"Age mean={age_mean:.2f}, std={age_std:.2f}", flush=True)

# 训练：RandomResizedCrop 防止视盘形变
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        (IMAGE_SIZE, IMAGE_SIZE), scale=(0.85, 1.0), ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.04),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 评测：Resize + CenterCrop 保比例
eval_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.1)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class FundusDataset(Dataset):
    def __init__(self, df, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            with Image.open(row["_path"]) as im:
                image = self.transform(im.convert("RGB"))
        except Exception:
            image = torch.zeros((3, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float32)

        age = (float(row["Age"]) - age_mean) / age_std
        age = torch.tensor(age, dtype=torch.float32)

        if self.is_test:
            return image, age

        label = torch.tensor(float(row["True_Label"]), dtype=torch.float32)
        return image, age, label


# 稳定版：单进程加载，绝不崩溃
loader_options = {
    "num_workers": NUM_WORKERS,
    "pin_memory": use_amp,
}

train_loader = DataLoader(
    FundusDataset(train_df, train_transform),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
    **loader_options
)
valid_loader = DataLoader(
    FundusDataset(valid_df, eval_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    **loader_options
)
test_loader = DataLoader(
    FundusDataset(test_df, eval_transform, is_test=True),
    batch_size=BATCH_SIZE, shuffle=False,
    **loader_options
)

print("Train batches:", len(train_loader), flush=True)
print("Valid batches:", len(valid_loader), flush=True)
print("Test batches:", len(test_loader), flush=True)

# -------------------- 5. 模型 --------------------
print("\n[STEP 4] 构造模型", flush=True)


class GlaucomaNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        if pretrained:
            weights = models.EfficientNet_B0_Weights.DEFAULT
            self.backbone = models.efficientnet_b0(weights=weights)
        else:
            self.backbone = models.efficientnet_b0(weights=None)
        feature_dim = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(feature_dim + 1, 1)

    def forward(self, image, age):
        features = self.backbone.features(image)
        features = self.backbone.avgpool(features)
        features = torch.flatten(features, 1)
        features = torch.cat([features, age.unsqueeze(1)], dim=1)
        return self.backbone.classifier(features).squeeze(1)


def build_model():
    # 1) torchvision 预训练
    try:
        m = GlaucomaNet(pretrained=True)
        print("Loaded ImageNet pretrained weights (torchvision).", flush=True)
        return m, True
    except Exception as e:
        print(f"torchvision pretrained download failed: {e}", flush=True)

    # 2) 本地预训练权重
    if LOCAL_PRETRAINED is not None and Path(LOCAL_PRETRAINED).exists():
        try:
            m = GlaucomaNet(pretrained=False)
            state = torch.load(LOCAL_PRETRAINED, map_location="cpu",
                               weights_only=False)
            if "model_state_dict" in state:
                state = state["model_state_dict"]
            missing, unexpected = m.load_state_dict(state, strict=False)
            print(f"Loaded local pretrained weights. "
                  f"missing={len(missing)}, unexpected={len(unexpected)}",
                  flush=True)
            return m, True
        except Exception as e:
            print(f"Local pretrained load failed: {e}", flush=True)

    # 3) 从零训练
    print("Falling back to training from scratch.", flush=True)
    return GlaucomaNet(pretrained=False), False


model, use_pretrained = build_model()
model = model.to(device)

is_parallel = False
if torch.cuda.device_count() > 1:
    print(f"启用 {torch.cuda.device_count()} 卡GPU并行训练", flush=True)
    model = nn.DataParallel(model)
    is_parallel = True

# 注意：稳定版不使用 channels_last（曾导致部分环境崩溃）

# 类别权重
n_pos = max(int(train_df["True_Label"].sum()), 1)
n_neg = len(train_df) - n_pos
positive_weight = min((n_neg / n_pos) ** 0.5, 8.0)
pos_weight = torch.tensor([positive_weight], device=device)
print(f"pos_weight = {positive_weight:.4f}", flush=True)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# 分层学习率
base_model = model.module if is_parallel else model
if use_pretrained:
    param_groups = [
        {"params": base_model.backbone.features.parameters(), "lr": LR_BACKBONE},
        {"params": base_model.backbone.classifier.parameters(), "lr": LR_HEAD},
    ]
else:
    param_groups = [
        {"params": base_model.parameters(), "lr": LR_HEAD},
    ]

optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / max(WARMUP_EPOCHS, 1)
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1 + np.cos(np.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# -------------------- 6. 断点续训 --------------------
start_epoch = 1
best_auc = -1.0
best_auprc = -1.0
history = []
epochs_no_improve = 0

if RESUME and LAST_PATH.exists():
    try:
        checkpoint = torch.load(LAST_PATH, map_location=device,
                                weights_only=False)
        base_model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_auc = float(checkpoint.get("best_auc", -1.0))
        best_auprc = float(checkpoint.get("best_auprc", -1.0))
        history = checkpoint.get("history", [])
        epochs_no_improve = int(checkpoint.get("epochs_no_improve", 0))
        print(f"Resuming from epoch {start_epoch}", flush=True)
    except Exception as e:
        print("Checkpoint could not be resumed; starting fresh:", e, flush=True)

if start_epoch > EPOCHS:
    print("Checkpoint already reached configured EPOCHS.", flush=True)

# -------------------- 7. 训练与验证 --------------------
print("\n[STEP 5] 开始训练", flush=True)

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    loss_sum = 0.0
    seen = 0

    print(f"\n{'=' * 20} Epoch {epoch}/{EPOCHS} {'=' * 20}", flush=True)
    progress = tqdm(train_loader, desc=f"Train {epoch}/{EPOCHS}",
                    dynamic_ncols=True)

    for step, (images, ages, labels) in enumerate(progress, 1):
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(images, ages)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        n = labels.size(0)
        loss_sum += loss.item() * n
        seen += n
        progress.set_postfix(
            loss=f"{loss_sum / max(seen, 1):.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.1e}"
        )

    train_loss = loss_sum / max(seen, 1)

    # 验证
    model.eval()
    val_probs, val_labels = [], []
    val_loss_sum, val_seen = 0.0, 0

    with torch.inference_mode():
        progress = tqdm(valid_loader, desc="Validation", dynamic_ncols=True)
        for images, ages, labels in progress:
            images = images.to(device, non_blocking=True)
            ages = ages.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(images, ages)
                loss = criterion(logits, labels)

            probs = torch.sigmoid(logits.float())
            val_probs.extend(probs.cpu().numpy().tolist())
            val_labels.extend(labels.cpu().numpy().tolist())
            val_loss_sum += loss.item() * labels.size(0)
            val_seen += labels.size(0)

    val_loss = val_loss_sum / max(val_seen, 1)
    val_auc = roc_auc_score(val_labels, val_probs)
    val_auprc = average_precision_score(val_labels, val_probs)
    elapsed_min = (time.time() - epoch_start) / 60

    print(
        f"\nEpoch {epoch}/{EPOCHS} | TrainLoss={train_loss:.4f} "
        f"| ValLoss={val_loss:.4f} | AUROC={val_auc:.5f} "
        f"| AUPRC={val_auprc:.5f} | Time={elapsed_min:.1f} min",
        flush=True
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": val_loss,
        "AUROC": val_auc,
        "AUPRC": val_auprc,
        "minutes": elapsed_min,
        "lr_backbone": optimizer.param_groups[0]["lr"],
    })
    pd.DataFrame(history).to_csv(LOG_PATH, index=False)

    # 保存最佳
    improved = np.isfinite(val_auc) and val_auc > best_auc
    if improved:
        best_auc = float(val_auc)
        best_auprc = float(val_auprc)
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": base_model.state_dict(),
            "valid_auc": float(val_auc),
            "valid_auprc": float(val_auprc),
            "age_mean": age_mean,
            "age_std": age_std,
            "image_size": IMAGE_SIZE,
            "pretrained": use_pretrained,
        }, BEST_PATH)
        print("Saved best model:", BEST_PATH, flush=True)
    else:
        epochs_no_improve += 1
        print(f"No improvement ({epochs_no_improve}/{PATIENCE})", flush=True)

    # 完整断点
    torch.save({
        "epoch": epoch,
        "model_state_dict": base_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_auc": best_auc,
        "best_auprc": best_auprc,
        "history": history,
        "epochs_no_improve": epochs_no_improve,
    }, LAST_PATH)

    scheduler.step()
    gc.collect()
    if use_amp:
        torch.cuda.empty_cache()

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.", flush=True)
        break

# -------------------- 8. 测试集预测（含 TTA） --------------------
print("\n[STEP 6] 加载最佳模型进行预测", flush=True)

if not BEST_PATH.exists():
    raise RuntimeError("No best model checkpoint exists.")

best_checkpoint = torch.load(BEST_PATH, map_location=device, weights_only=False)
base_model.load_state_dict(best_checkpoint["model_state_dict"])
base_model.eval()
model = base_model  # 单卡推理；如需并行可再包 DataParallel

test_probs = []
with torch.inference_mode():
    progress = tqdm(test_loader, desc="Predict test set", dynamic_ncols=True)
    for images, ages in progress:
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits1 = model(images, ages)
            if USE_TTA:
                logits2 = model(torch.flip(images, dims=[3]), ages)
            else:
                logits2 = None

        p1 = torch.sigmoid(logits1.float())
        if logits2 is not None:
            p2 = torch.sigmoid(logits2.float())
            probs = (p1 + p2) / 2.0
        else:
            probs = p1

        test_probs.extend(probs.cpu().numpy().tolist())

# -------------------- 9. 官方三列提交文件 --------------------
test_probs_np = np.array(test_probs)
pred_labels = (test_probs_np >= 0.5).astype(int)

submission = pd.DataFrame({
    "Image_name": test_df["Image_name"].astype(str),
    "Probability": test_probs_np.astype(float),
    "Pred_Label": pred_labels.astype(int),
})
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

print("\n" + "=" * 65, flush=True)
print("Training and test prediction finished.", flush=True)
print("Best epoch:", best_checkpoint["epoch"], flush=True)
print("Best validation AUROC:", best_checkpoint["valid_auc"], flush=True)
print("Best validation AUPRC:", best_checkpoint["valid_auprc"], flush=True)
print("Pretrained used:", best_checkpoint.get("pretrained"), flush=True)
print("TTA enabled:", USE_TTA, flush=True)
print("Training log:", LOG_PATH, flush=True)
print("Best model:", BEST_PATH, flush=True)
print("Resume checkpoint:", LAST_PATH, flush=True)
print("Submission:", SUBMISSION_PATH, flush=True)
print("Submission shape:", submission.shape, flush=True)
print(submission.head(), flush=True)

In [ ]:
import shutil
from pathlib import Path
import os

WORK = Path("/kaggle/working")
backup_dir = WORK / "checkpoints"
backup_dir.mkdir(exist_ok=True)

for name in ["glaucoma_last.pth", "glaucoma_best.pth", "training_log.csv",
             "image_index_cache.pkl"]:
    src = WORK / name
    if src.exists():
        dst = backup_dir / name
        shutil.copy(src, dst)
        print(f"backed up: {name} ({src.stat().st_size/1024/1024:.1f} MB)")
    else:
        print(f"not found yet: {name}")

In [ ]:
from pathlib import Path
for p in ["/kaggle/working/glaucoma_last.pth",
          "/kaggle/working/glaucoma_best.pth"]:
    p = Path(p)
    if p.exists():
        p.unlink()
        print("removed", p)
# image_index_cache.pkl 保留，可以复用
print("done")

In [ ]:
# ============================================================
# Glaucoma Competition - Fixed Stable Version (single T4)
# 修复：单GPU | NUM_WORKERS=2 | pin_memory=False | EPOCHS=8 | PATIENCE=3
# 保留：预训练 | warmup+cosine | TTA | 断点续训 | 坏图容错
# ============================================================

# ---- 单 GPU，必须在 import torch 之前 ----
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import time
import random
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# -------------------- 1. Configuration --------------------
DATA_ROOT = Path("/kaggle/input/datasets/dididoudou/glaucoma-data")
WORK = Path("/kaggle/working")

TRAIN_CSV = DATA_ROOT / "train.csv"
VALID_CSV = DATA_ROOT / "valid.csv"
TEST_CSV = DATA_ROOT / "test.csv"

# 本地预训练权重（可选）
LOCAL_PRETRAINED = None

# ---- 稳定 + 速度平衡配置 ----
IMAGE_SIZE = 320
BATCH_SIZE = 32
EPOCHS = 8                # 当前配额下 8 epoch 足够
WARMUP_EPOCHS = 1
PATIENCE = 3              # 连续 3 个 epoch 无提升就早停
NUM_WORKERS = 2           # 关键提速
LR_BACKBONE = 3e-5
LR_HEAD = 3e-4
SEED = 42

USE_TTA = True

BEST_PATH = WORK / "glaucoma_best.pth"
LAST_PATH = WORK / "glaucoma_last.pth"
LOG_PATH = WORK / "training_log.csv"
SUBMISSION_PATH = WORK / "submission.csv"
INDEX_CACHE = WORK / "image_index_cache.pkl"

RESUME = True

# ---- 随机种子 ----
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("=" * 65, flush=True)
print("Device:", device, flush=True)
if use_amp:
    print("可见GPU数量：", torch.cuda.device_count(), flush=True)
    print("GPU：", torch.cuda.get_device_name(0), flush=True)
    print("显存：",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB", flush=True)
print(f"Image size: {IMAGE_SIZE} | Batch: {BATCH_SIZE} | "
      f"Epochs: {EPOCHS} | Workers: {NUM_WORKERS} | TTA: {USE_TTA}", flush=True)

# -------------------- 2. Read CSV --------------------
print("[STEP 1] 读取 CSV", flush=True)
for p in [TRAIN_CSV, VALID_CSV, TEST_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"CSV not found: {p}")

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)
test_df = pd.read_csv(TEST_CSV)

for name, df in [("train", train_df), ("valid", valid_df)]:
    needed = {"Image_name", "Age", "True_Label"}
    if not needed.issubset(df.columns):
        raise ValueError(f"{name}.csv columns unexpected: {df.columns.tolist()}")

if not {"Image_name", "Age"}.issubset(test_df.columns):
    raise ValueError(f"test.csv columns unexpected: {test_df.columns.tolist()}")

train_df["True_Label"] = train_df["True_Label"].astype(int)
valid_df["True_Label"] = valid_df["True_Label"].astype(int)

age_median = float(train_df["Age"].median())
print("Age median:", age_median, flush=True)
for df in [train_df, valid_df, test_df]:
    df["Age"] = df["Age"].fillna(age_median)

print(f"Train: {len(train_df)}, positives: {int(train_df['True_Label'].sum())}",
      flush=True)
print(f"Valid: {len(valid_df)}, positives: {int(valid_df['True_Label'].sum())}",
      flush=True)
print(f"Test: {len(test_df)}", flush=True)
gc.collect()

# -------------------- 3. 图片索引 --------------------
print("\n[STEP 2] 扫描/加载图片索引", flush=True)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

image_index = None
if INDEX_CACHE.exists():
    try:
        with open(INDEX_CACHE, "rb") as f:
            image_index = pickle.load(f)
        print("Cache loaded, images:", len(image_index), flush=True)
    except Exception as e:
        print("Cache broken, rebuild:", e, flush=True)
        INDEX_CACHE.unlink()
        image_index = None

if image_index is None:
    all_image_paths = []
    scan_root = "/kaggle/input/datasets/dididoudou"
    for root, dirs, files in os.walk(scan_root):
        for filename in files:
            if Path(filename).suffix.lower() in IMAGE_EXTS:
                all_image_paths.append(os.path.join(root, filename))
    print("Image files found:", len(all_image_paths), flush=True)

    image_index = {}
    for path in tqdm(all_image_paths, desc="Indexing", dynamic_ncols=True):
        key = os.path.basename(path).lower()
        image_index.setdefault(key, path)

    with open(INDEX_CACHE, "wb") as f:
        pickle.dump(image_index, f)
    print("Index saved.", flush=True)

print("Total images in index:", len(image_index), flush=True)


def find_image(filename):
    return image_index.get(os.path.basename(str(filename)).lower())


train_df["_path"] = train_df["Image_name"].map(find_image)
valid_df["_path"] = valid_df["Image_name"].map(find_image)
test_df["_path"] = test_df["Image_name"].map(find_image)

for name, df in [("Train", train_df), ("Valid", valid_df), ("Test", test_df)]:
    found = int(df["_path"].notna().sum())
    print(f"{name} image match: {found}/{len(df)}", flush=True)
    if found != len(df):
        missing = df.loc[df["_path"].isna(), "Image_name"].head(10).tolist()
        raise RuntimeError(f"{name} images missing. Examples: {missing}")
gc.collect()

# -------------------- 4. Dataset --------------------
print("\n[STEP 3] 构造 Dataset / DataLoader", flush=True)

age_mean = float(train_df["Age"].mean())
age_std = max(float(train_df["Age"].std()), 1.0)
print(f"Age mean={age_mean:.2f}, std={age_std:.2f}", flush=True)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        (IMAGE_SIZE, IMAGE_SIZE), scale=(0.85, 1.0), ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.04),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.1)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class FundusDataset(Dataset):
    def __init__(self, df, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            with Image.open(row["_path"]) as im:
                image = self.transform(im.convert("RGB"))
        except Exception:
            image = torch.zeros((3, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float32)

        age = (float(row["Age"]) - age_mean) / age_std
        age = torch.tensor(age, dtype=torch.float32)

        if self.is_test:
            return image, age

        label = torch.tensor(float(row["True_Label"]), dtype=torch.float32)
        return image, age, label


# 关键：pin_memory=False 解决 RAM 爆满
loader_options = {
    "num_workers": NUM_WORKERS,
    "pin_memory": False,
}
if NUM_WORKERS > 0:
    loader_options["persistent_workers"] = True
    loader_options["prefetch_factor"] = 2

train_loader = DataLoader(
    FundusDataset(train_df, train_transform),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
    **loader_options
)
valid_loader = DataLoader(
    FundusDataset(valid_df, eval_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    **loader_options
)
test_loader = DataLoader(
    FundusDataset(test_df, eval_transform, is_test=True),
    batch_size=BATCH_SIZE, shuffle=False,
    **loader_options
)

print("Train batches:", len(train_loader), flush=True)
print("Valid batches:", len(valid_loader), flush=True)
print("Test batches:", len(test_loader), flush=True)

# -------------------- 5. 模型 --------------------
print("\n[STEP 4] 构造模型", flush=True)


class GlaucomaNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        if pretrained:
            weights = models.EfficientNet_B0_Weights.DEFAULT
            self.backbone = models.efficientnet_b0(weights=weights)
        else:
            self.backbone = models.efficientnet_b0(weights=None)
        feature_dim = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(feature_dim + 1, 1)

    def forward(self, image, age):
        features = self.backbone.features(image)
        features = self.backbone.avgpool(features)
        features = torch.flatten(features, 1)
        features = torch.cat([features, age.unsqueeze(1)], dim=1)
        return self.backbone.classifier(features).squeeze(1)


def build_model():
    try:
        m = GlaucomaNet(pretrained=True)
        print("Loaded ImageNet pretrained weights.", flush=True)
        return m, True
    except Exception as e:
        print(f"Pretrained download failed: {e}", flush=True)

    if LOCAL_PRETRAINED is not None and Path(LOCAL_PRETRAINED).exists():
        try:
            m = GlaucomaNet(pretrained=False)
            state = torch.load(LOCAL_PRETRAINED, map_location="cpu",
                               weights_only=False)
            if "model_state_dict" in state:
                state = state["model_state_dict"]
            m.load_state_dict(state, strict=False)
            print("Loaded local pretrained weights.", flush=True)
            return m, True
        except Exception as e:
            print(f"Local pretrained failed: {e}", flush=True)

    print("Training from scratch.", flush=True)
    return GlaucomaNet(pretrained=False), False


model, use_pretrained = build_model()
model = model.to(device)

# 单 GPU：不做 DataParallel
is_parallel = False
base_model = model

# 类别权重
n_pos = max(int(train_df["True_Label"].sum()), 1)
n_neg = len(train_df) - n_pos
positive_weight = min((n_neg / n_pos) ** 0.5, 8.0)
pos_weight = torch.tensor([positive_weight], device=device)
print(f"pos_weight = {positive_weight:.4f}", flush=True)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# 分层学习率
if use_pretrained:
    param_groups = [
        {"params": base_model.backbone.features.parameters(), "lr": LR_BACKBONE},
        {"params": base_model.backbone.classifier.parameters(), "lr": LR_HEAD},
    ]
else:
    param_groups = [
        {"params": base_model.parameters(), "lr": LR_HEAD},
    ]

optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / max(WARMUP_EPOCHS, 1)
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1 + np.cos(np.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# -------------------- 6. 断点续训 --------------------
start_epoch = 1
best_auc = -1.0
best_auprc = -1.0
history = []
epochs_no_improve = 0

if RESUME and LAST_PATH.exists():
    try:
        checkpoint = torch.load(LAST_PATH, map_location=device,
                                weights_only=False)
        base_model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_auc = float(checkpoint.get("best_auc", -1.0))
        best_auprc = float(checkpoint.get("best_auprc", -1.0))
        history = checkpoint.get("history", [])
        epochs_no_improve = int(checkpoint.get("epochs_no_improve", 0))
        print(f"Resuming from epoch {start_epoch}", flush=True)
    except Exception as e:
        print("Resume failed, starting fresh:", e, flush=True)

if start_epoch > EPOCHS:
    print("Already reached EPOCHS.", flush=True)

# -------------------- 7. 训练 --------------------
print("\n[STEP 5] 开始训练", flush=True)

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    loss_sum = 0.0
    seen = 0

    print(f"\n{'=' * 20} Epoch {epoch}/{EPOCHS} {'=' * 20}", flush=True)
    progress = tqdm(train_loader, desc=f"Train {epoch}/{EPOCHS}",
                    dynamic_ncols=True)

    for step, (images, ages, labels) in enumerate(progress, 1):
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(images, ages)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        n = labels.size(0)
        loss_sum += loss.item() * n
        seen += n
        progress.set_postfix(
            loss=f"{loss_sum / max(seen, 1):.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.1e}"
        )

    train_loss = loss_sum / max(seen, 1)

    # 验证
    model.eval()
    val_probs, val_labels = [], []
    val_loss_sum, val_seen = 0.0, 0

    with torch.inference_mode():
        progress = tqdm(valid_loader, desc="Validation", dynamic_ncols=True)
        for images, ages, labels in progress:
            images = images.to(device, non_blocking=True)
            ages = ages.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(images, ages)
                loss = criterion(logits, labels)

            probs = torch.sigmoid(logits.float())
            val_probs.extend(probs.cpu().numpy().tolist())
            val_labels.extend(labels.cpu().numpy().tolist())
            val_loss_sum += loss.item() * labels.size(0)
            val_seen += labels.size(0)

    val_loss = val_loss_sum / max(val_seen, 1)
    val_auc = roc_auc_score(val_labels, val_probs)
    val_auprc = average_precision_score(val_labels, val_probs)
    elapsed_min = (time.time() - epoch_start) / 60

    print(
        f"\nEpoch {epoch}/{EPOCHS} | TrainLoss={train_loss:.4f} "
        f"| ValLoss={val_loss:.4f} | AUROC={val_auc:.5f} "
        f"| AUPRC={val_auprc:.5f} | Time={elapsed_min:.1f} min",
        flush=True
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": val_loss,
        "AUROC": val_auc,
        "AUPRC": val_auprc,
        "minutes": elapsed_min,
        "lr_backbone": optimizer.param_groups[0]["lr"],
    })
    pd.DataFrame(history).to_csv(LOG_PATH, index=False)

    improved = np.isfinite(val_auc) and val_auc > best_auc
    if improved:
        best_auc = float(val_auc)
        best_auprc = float(val_auprc)
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": base_model.state_dict(),
            "valid_auc": float(val_auc),
            "valid_auprc": float(val_auprc),
            "age_mean": age_mean,
            "age_std": age_std,
            "image_size": IMAGE_SIZE,
            "pretrained": use_pretrained,
        }, BEST_PATH)
        print("Saved best model:", BEST_PATH, flush=True)
    else:
        epochs_no_improve += 1
        print(f"No improvement ({epochs_no_improve}/{PATIENCE})", flush=True)

    torch.save({
        "epoch": epoch,
        "model_state_dict": base_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_auc": best_auc,
        "best_auprc": best_auprc,
        "history": history,
        "epochs_no_improve": epochs_no_improve,
    }, LAST_PATH)

    scheduler.step()
    gc.collect()
    if use_amp:
        torch.cuda.empty_cache()

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.", flush=True)
        break

# -------------------- 8. 预测 --------------------
print("\n[STEP 6] 加载最佳模型预测", flush=True)

if not BEST_PATH.exists():
    raise RuntimeError("No best model checkpoint exists.")

best_checkpoint = torch.load(BEST_PATH, map_location=device, weights_only=False)
base_model.load_state_dict(best_checkpoint["model_state_dict"])
base_model.eval()

test_probs = []
with torch.inference_mode():
    progress = tqdm(test_loader, desc="Predict", dynamic_ncols=True)
    for images, ages in progress:
        images = images.to(device, non_blocking=True)
        ages = ages.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits1 = base_model(images, ages)
            if USE_TTA:
                logits2 = base_model(torch.flip(images, dims=[3]), ages)
            else:
                logits2 = None

        p1 = torch.sigmoid(logits1.float())
        if logits2 is not None:
            probs = (p1 + torch.sigmoid(logits2.float())) / 2.0
        else:
            probs = p1

        test_probs.extend(probs.cpu().numpy().tolist())

# -------------------- 9. 提交 --------------------
test_probs_np = np.array(test_probs)
pred_labels = (test_probs_np >= 0.5).astype(int)

submission = pd.DataFrame({
    "Image_name": test_df["Image_name"].astype(str),
    "Probability": test_probs_np.astype(float),
    "Pred_Label": pred_labels.astype(int),
})
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

print("\n" + "=" * 65, flush=True)
print("Training and prediction finished.", flush=True)
print("Best epoch:", best_checkpoint["epoch"], flush=True)
print("Best AUROC:", best_checkpoint["valid_auc"], flush=True)
print("Best AUPRC:", best_checkpoint["valid_auprc"], flush=True)
print("Pretrained:", best_checkpoint.get("pretrained"), flush=True)
print("TTA:", USE_TTA, flush=True)
print("Submission:", SUBMISSION_PATH, flush=True)
print("Shape:", submission.shape, flush=True)
print(submission.head(), flush=True)

In [ ]:
import shutil
from pathlib import Path
WORK = Path("/kaggle/working")
backup_dir = WORK / "checkpoints"
backup_dir.mkdir(exist_ok=True)

for name in ["glaucoma_last.pth", "glaucoma_best.pth", "training_log.csv"]:
    src = WORK / name
    if src.exists():
        shutil.copy(src, backup_dir / name)
        print("✅ 备份成功:", name)
    else:
        print("❌ 未找到:", name)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import pandas as pd, numpy as np, torch, torch.nn as nn, time, warnings
from pathlib import Path
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ===== 1. 配置 =====
DATA_ROOT = Path("/kaggle/input/datasets/dididoudou/glaucoma-data")
WORK = Path("/kaggle/working")
TRAIN_CSV, VALID_CSV, TEST_CSV = DATA_ROOT/"train.csv", DATA_ROOT/"valid.csv", DATA_ROOT/"test.csv"
IMAGE_SIZE, BATCH_SIZE, EPOCHS = 320, 32, 1  # 强制只跑 1 个 epoch
LR_HEAD, LR_BACKBONE, SEED = 3e-4, 3e-5, 42
BEST_PATH, SUBMISSION_PATH = WORK/"glaucoma_best.pth", WORK/"submission.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
torch.manual_seed(SEED)

# ===== 2. 读数据 + 自动扫描图片 =====
print("[1/4] 读取 CSV 并扫描图片...", flush=True)
train_df, valid_df, test_df = pd.read_csv(TRAIN_CSV), pd.read_csv(VALID_CSV), pd.read_csv(TEST_CSV)
for df in [train_df, valid_df, test_df]:
    df["Age"] = df["Age"].fillna(train_df["Age"].median())
    df["True_Label"] = df["True_Label"].astype(int) if "True_Label" in df.columns else 0

image_index = {}
for root_dir, _, files in os.walk("/kaggle/input"):
    for f in files:
        if Path(f).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}:
            image_index[f.lower()] = os.path.join(root_dir, f)

for df in [train_df, valid_df, test_df]:
    df["_path"] = df["Image_name"].map(lambda x: image_index.get(str(x).lower()))
assert train_df["_path"].notna().all() and test_df["_path"].notna().all(), "图片缺失！"
print(f"图片匹配完成: Train {len(train_df)}, Test {len(test_df)}", flush=True)

# ===== 3. 数据集与模型 =====
age_mean, age_std = train_df["Age"].mean(), max(train_df["Age"].std(), 1.0)
train_tf = transforms.Compose([
    transforms.RandomResizedCrop((IMAGE_SIZE, IMAGE_SIZE), scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ColorJitter(0.08, 0.08, 0.04),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
eval_tf = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE*1.1)), transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
class DS(Dataset):
    def __init__(self, df, tf, is_test=False): self.df, self.tf, self.is_test = df.reset_index(drop=True), tf, is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:
            with Image.open(r["_path"]) as im: img = self.tf(im.convert("RGB"))
        except: img = torch.zeros((3, IMAGE_SIZE, IMAGE_SIZE))
        age = torch.tensor((float(r["Age"])-age_mean)/age_std, dtype=torch.float32)
        if self.is_test: return img, age
        return img, age, torch.tensor(float(r["True_Label"]), dtype=torch.float32)

train_loader = DataLoader(DS(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=False)
valid_loader = DataLoader(DS(valid_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)
test_loader = DataLoader(DS(test_df, eval_tf, is_test=True), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)

class GlaucomaNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        dim = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(dim + 1, 1)
    def forward(self, image, age):
        f = self.backbone.features(image); f = self.backbone.avgpool(f)
        f = torch.flatten(f, 1); f = torch.cat([f, age.unsqueeze(1)], dim=1)
        return self.backbone.classifier(f).squeeze(1)

model = GlaucomaNet().to(device)
pos_weight = torch.tensor([min((len(train_df)-train_df["True_Label"].sum()) / max(train_df["True_Label"].sum(), 1)**0.5, 8.0)], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW([
    {"params": model.backbone.features.parameters(), "lr": LR_BACKBONE},
    {"params": model.backbone.classifier.parameters(), "lr": LR_HEAD}
], weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# ===== 4. 开始 1 个 Epoch 的极速训练 =====
print(f"\n[2/4] 开始救急训练 (1 个 Epoch)...", flush=True)
model.train()
for step, (imgs, ages, labels) in enumerate(tqdm(train_loader, desc="Train"), 1):
    imgs, ages, labels = imgs.to(device), ages.to(device), labels.to(device)
    optimizer.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast(enabled=use_amp):
        loss = criterion(model(imgs, ages), labels)
    scaler.scale(loss).backward()
    scaler.step(optimizer); scaler.update()

# 验证一下 Epoch 1 的成绩
model.eval(); val_probs, val_labels = [], []
with torch.inference_mode():
    for imgs, ages, labels in tqdm(valid_loader, desc="Valid"):
        imgs, ages, labels = imgs.to(device), ages.to(device), labels.to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs, ages)
        val_probs.extend(torch.sigmoid(logits.float()).cpu().numpy().tolist())
        val_labels.extend(labels.cpu().numpy().tolist())
val_auc = roc_auc_score(val_labels, val_probs)
print(f"\n✅ Epoch 1 完成！AUROC: {val_auc:.5f}", flush=True)

# 保存模型
torch.save({"model_state_dict": model.state_dict(), "age_mean": age_mean, "age_std": age_std, "image_size": IMAGE_SIZE}, BEST_PATH)
print("模型已保存至:", BEST_PATH, flush=True)

# ===== 5. 极速预测并生成 CSV =====
print("\n[3/4] 开始预测测试集...", flush=True)
model.eval(); test_probs = []
with torch.inference_mode():
    for imgs, ages in tqdm(test_loader, desc="Predict"):
        imgs, ages = imgs.to(device), ages.to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            l1 = model(imgs, ages)
            l2 = model(torch.flip(imgs, dims=[3]), ages)  # TTA
        p = (torch.sigmoid(l1.float()) + torch.sigmoid(l2.float())) / 2
        test_probs.extend(p.cpu().numpy().tolist())

sub = pd.DataFrame({
    "Image_name": test_df["Image_name"].astype(str),
    "Probability": np.array(test_probs),
    "Pred_Label": (np.array(test_probs) >= 0.5).astype(int)
})
sub.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")
print(f"\n[4/4] 🎉 成功！Submission 已保存，形状: {sub.shape}", flush=True)
print(sub.head())

In [1]:
!rm -f /kaggle/working/*.pth
!rm -f /kaggle/working/*.csv
print("旧文件清理完毕！")

旧文件清理完毕！


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # 单卡，省额度
import pandas as pd, numpy as np, torch, torch.nn as nn, time, warnings, random, gc
from pathlib import Path
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import functional as TF
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ===== 1. 极限保命配置 =====
DATA_ROOT = Path("/kaggle/input/datasets/dididoudou/glaucoma-data")
WORK = Path("/kaggle/working")
TRAIN_CSV, VALID_CSV, TEST_CSV = DATA_ROOT/"train.csv", DATA_ROOT/"valid.csv", DATA_ROOT/"test.csv"
BEST_PATH = WORK/"glaucoma_best.pth"
LATEST_PATH = WORK/"latest_checkpoint.pth"  # 断点续训备份
SUBMISSION_PATH = WORK/"submission.csv"

IMAGE_SIZE, BATCH_SIZE = 320, 32
MAX_EPOCHS = 3               # 锁定3轮，保证5.5h内绝对跑完
PATIENCE = 2                 # 连续2轮不提升就早停
LR_HEAD, LR_BACKBONE, SEED = 5e-4, 5e-5, 42
TIME_LIMIT_HOURS = 4.5       # 硬性时间熔断：超过4.5h强制停止训练生成CSV

# 【关键修复1】MixUp 置为 0，避免逻辑冲突和多余耗时
MIXUP_PROB = 0.0  
MIXUP_ALPHA = 0.2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(SEED)

# ===== 2. 读数据 + 自动扫描 =====
print("[1/5] 读取 CSV 并扫描图片...", flush=True)
train_df, valid_df, test_df = pd.read_csv(TRAIN_CSV), pd.read_csv(VALID_CSV), pd.read_csv(TEST_CSV)
for df in [train_df, valid_df, test_df]:
    df["Age"] = df["Age"].fillna(train_df["Age"].median())
train_df["True_Label"] = train_df["True_Label"].astype(int)
valid_df["True_Label"] = valid_df["True_Label"].astype(int)

image_index = {}
for root_dir, _, files in os.walk("/kaggle/input"):
    for f in files:
        if Path(f).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}:
            image_index[f.lower()] = os.path.join(root_dir, f)

for df in [train_df, valid_df, test_df]:
    df["_path"] = df["Image_name"].map(lambda x: image_index.get(str(x).lower()))
assert train_df["_path"].notna().all() and test_df["_path"].notna().all(), "图片缺失！"
print(f"图片匹配完成: Train {len(train_df)}, Test {len(test_df)}", flush=True)

# ===== 3. 数据集与增强 =====
age_mean, age_std = train_df["Age"].mean(), max(train_df["Age"].std(), 1.0)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop((IMAGE_SIZE, IMAGE_SIZE), scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10, fill=0),
    transforms.ColorJitter(0.08, 0.08, 0.04),
    transforms.RandomAdjustSharpness(sharpness_factor=1.2, p=0.3),  # 眼底边缘增强
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE*1.1)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class DS(Dataset):
    def __init__(self, df, tf, is_test=False):
        self.df, self.tf, self.is_test = df.reset_index(drop=True), tf, is_test
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:
            with Image.open(r["_path"]) as im:
                img = self.tf(im.convert("RGB"))
        except Exception:
            img = torch.zeros((3, IMAGE_SIZE, IMAGE_SIZE))
        age = torch.tensor((float(r["Age"])-age_mean)/age_std, dtype=torch.float32)
        if self.is_test:
            return img, age
        return img, age, torch.tensor(float(r["True_Label"]), dtype=torch.float32)

# 【关键修复2】pin_memory=False 防止 RAM 爆满，保留 num_workers=2 保证速度
train_loader = DataLoader(
    DS(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=False, drop_last=True, prefetch_factor=2
)
valid_loader = DataLoader(
    DS(valid_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=False, prefetch_factor=2
)
test_loader = DataLoader(
    DS(test_df, eval_tf, is_test=True), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=False, prefetch_factor=2
)

# ===== 4. 模型与损失：升级B1 =====
class GlaucomaNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)
        dim = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        
        self.age_head = nn.Sequential(
            nn.Linear(1, 32),
            nn.GELU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.2)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(dim + 32, 1)
        )
    
    def forward(self, image, age):
        f = self.backbone(image)
        a = self.age_head(age.unsqueeze(1))
        return self.classifier(torch.cat([f, a], dim=1)).squeeze(1)

model = GlaucomaNet().to(device)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight
    
    def forward(self, logits, labels):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, labels, pos_weight=self.pos_weight, reduction="none"
        )
        p_t = torch.exp(-bce)
        return ((1 - p_t) ** self.gamma * bce).mean()

pos_num = max(train_df["True_Label"].sum().item(), 1)
neg_num = len(train_df) - pos_num
pos_weight = torch.tensor([min(neg_num / pos_num, 8.0)], device=device)

criterion = FocalLoss(gamma=2.0, pos_weight=pos_weight)

optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
    {"params": model.age_head.parameters(), "lr": LR_HEAD},
    {"params": model.classifier.parameters(), "lr": LR_HEAD}
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[LR_BACKBONE, LR_HEAD, LR_HEAD],
    steps_per_epoch=len(train_loader),
    epochs=MAX_EPOCHS,
    pct_start=0.15
)

scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# ===== 5. 生成提交文件 =====
def generate_submission(model_to_use):
    model_to_use.eval()
    test_probs = []
    tta_augs = [
        lambda x: x,
        lambda x: TF.hflip(x),
        lambda x: TF.rotate(x, 10, fill=0),
        lambda x: TF.rotate(x, -10, fill=0),
    ]
    with torch.inference_mode():
        for imgs, ages in tqdm(test_loader, desc="[TTA预测]", leave=False):
            imgs, ages = imgs.to(device), ages.to(device)
            batch_probs = 0.0
            for aug in tta_augs:
                with torch.cuda.amp.autocast(enabled=use_amp):
                    logits = model_to_use(aug(imgs), ages)
                batch_probs += torch.sigmoid(logits.float())
            test_probs.extend((batch_probs / len(tta_augs)).cpu().numpy().tolist())
    
    sub = pd.DataFrame({
        "Image_name": test_df["Image_name"].astype(str),
        "Probability": np.array(test_probs),
        "Pred_Label": (np.array(test_probs) >= 0.5).astype(int)
    })
    sub.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")
    print(f"✅ Submission 已覆盖保存！形状: {sub.shape}", flush=True)

# ===== 6. 保命训练循环 + 断点续训 =====
print(f"\n[2/5] 开始安全训练 (最多 {MAX_EPOCHS} 轮)...", flush=True)
start_time = time.time()
best_auc = 0.0
start_epoch = 1
patience_cnt = 0

if LATEST_PATH.exists():
    print("检测到中断备份，正在恢复训练状态...", flush=True)
    ckpt = torch.load(LATEST_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_auc = ckpt["best_auc"]
    patience_cnt = ckpt["patience_cnt"]
    print(f"恢复成功！从第 {start_epoch} 轮继续，历史最优AUROC: {best_auc:.5f}", flush=True)

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    if (time.time() - start_time) / 3600 > TIME_LIMIT_HOURS:
        print(f"\n⚠️ 触发4.5小时时间熔断，强制停止训练！", flush=True)
        break
    
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS} Train", dynamic_ncols=True, leave=False)
    
    for imgs, ages, labels in pbar:
        if (time.time() - start_time) / 3600 > TIME_LIMIT_HOURS:
            print("\n⚠️ 训练中触发时间熔断！", flush=True)
            break
        
        imgs, ages, labels = imgs.to(device), ages.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss = criterion(model(imgs, ages), labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    gc.collect()
    torch.cuda.empty_cache()
    
    # 验证
    model.eval()
    val_probs, val_labels = [], []
    with torch.inference_mode():
        for imgs, ages, labels in tqdm(valid_loader, desc=f"Epoch {epoch} Valid", leave=False):
            imgs, ages, labels = imgs.to(device), ages.to(device), labels.to(device)
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(imgs, ages)
            val_probs.extend(torch.sigmoid(logits.float()).cpu().numpy().tolist())
            val_labels.extend(labels.cpu().numpy().tolist())
    
    val_auc = roc_auc_score(val_labels, val_probs)
    elapsed = (time.time() - start_time) / 3600
    print(f"\nEpoch {epoch} | 验证集 AUROC: {val_auc:.5f} | 已用时: {elapsed:.2f}h", flush=True)
    
    # 保存最新备份
    torch.save({
        "epoch": epoch,
        "best_auc": best_auc,
        "patience_cnt": patience_cnt,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
    }, LATEST_PATH)
    
    # 保存最优模型并生成提交文件
    if val_auc > best_auc:
        best_auc = val_auc
        patience_cnt = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "age_mean": age_mean,
            "age_std": age_std,
            "image_size": IMAGE_SIZE
        }, BEST_PATH)
        print(f"🌟 发现新高！保存模型并立刻生成最新 CSV...", flush=True)
        generate_submission(model)
    else:
        patience_cnt += 1
        print(f"未提升，耐心值: {patience_cnt}/{PATIENCE}", flush=True)
        if patience_cnt >= PATIENCE:
            print(f"\n🛑 早停触发！", flush=True)
            break

# ===== 7. 最终兜底 =====
print("\n[3/5] 训练结束。最终兜底检查...", flush=True)
torch.cuda.empty_cache()
gc.collect()

if not SUBMISSION_PATH.exists():
    print("未生成 CSV，使用当前模型强制生成...", flush=True)
    generate_submission(model)
else:
    print(f"✅ 最终 CSV 已就绪！请前往右侧 Output 面板下载。", flush=True)
    print(f"当前最优验证 AUROC: {best_auc:.5f}", flush=True)
    print(f"总耗时: {(time.time()-start_time)/3600:.2f} 小时", flush=True)

[1/5] 读取 CSV 并扫描图片...
图片匹配完成: Train 51396, Test 5855
Downloading: "https://download.pytorch.org/models/efficientnet_b1-c27df63c.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1-c27df63c.pth


100%|██████████| 30.1M/30.1M [00:00<00:00, 116MB/s] 



[2/5] 开始安全训练 (最多 3 轮)...


Epoch 1/3 Train:   0%|          | 0/1606 [00:00<?, ?it/s]